# 📉 Customer Churn Prediction — EDA Notebook
**Notebook 01: Exploratory Data Analysis**

Covers Steps 2–6:
- Step 2: Convert `signup_date` to datetime
- Step 3: Separate features (Numerical / Categorical / Binary)
- Step 4: Target variable analysis
- Step 5: Numerical features — distributions, box plots, stats
- Step 6: Categorical & Binary features — counts, percentages


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# Style
plt.style.use('dark_background')
sns.set_palette(['#6C63FF', '#FF6584', '#43BCCD', '#F7B731', '#a29bfe', '#fd79a8'])

PALETTE = ['#6C63FF', '#FF6584']
FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)
print('Setup complete.')

---
## 📥 Load the Dataset

In [ ]:
df = pd.read_csv('../data/raw/customer_churn_1M.csv')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

---
## 🕐 Step 2: Convert signup_date to Datetime

In [ ]:
df['signup_date'] = pd.to_datetime(df['signup_date'])

print(f'dtype after conversion : {df["signup_date"].dtype}')
print(f'Earliest signup date   : {df["signup_date"].min()}')
print(f'Latest signup date     : {df["signup_date"].max()}')
date_range = df['signup_date'].max() - df['signup_date'].min()
print(f'Date range             : {date_range.days:,} days (~{date_range.days // 365} years)')

# Extract date parts for later feature engineering
df['signup_year']    = df['signup_date'].dt.year
df['signup_month']   = df['signup_date'].dt.month
df['signup_quarter'] = df['signup_date'].dt.quarter

print(f'\nYears present : {sorted(df["signup_year"].unique())}')

---
## 🗂️ Step 3: Separate Features

In [ ]:
NUMERICAL = [
    'age', 'annual_income', 'tenure', 'monthlycharges', 'totalcharges',
    'customer_satisfaction', 'num_complaints', 'num_service_calls',
    'late_payments', 'avg_monthly_gb', 'days_since_last_interaction', 'credit_score',
]
CATEGORICAL = [
    'gender', 'education', 'marital_status', 'contract', 'payment_method', 'paperless_billing',
]
BINARY = [
    'has_phone_service', 'has_internet_service', 'has_online_security',
    'has_online_backup', 'has_device_protection', 'has_tech_support',
    'has_streaming_tv', 'has_streaming_movies', 'senior_citizen',
]
TARGET = 'churn'

print(f'Numerical   ({len(NUMERICAL)}): {NUMERICAL}')
print(f'Categorical ({len(CATEGORICAL)}): {CATEGORICAL}')
print(f'Binary      ({len(BINARY)}): {BINARY}')
print(f'Target      : {TARGET}')

---
## 🎯 Step 4: Target Variable Analysis — Churn Distribution

In [ ]:
vc  = df[TARGET].value_counts()
pct = df[TARGET].value_counts(normalize=True) * 100

print('Churn Distribution:')
print(f'  No Churn (0): {vc[0]:>9,}  ({pct[0]:.2f}%)')
print(f'  Churn    (1): {vc[1]:>9,}  ({pct[1]:.2f}%)')
print()
print('OBSERVATION:')
print(f'Around {pct[0]:.1f}% of customers stayed while ~{pct[1]:.1f}% churned.')
print('This is a HIGHLY IMBALANCED dataset — use stratified splits,')
print('SMOTE/class_weight, and AUC/F1 as primary evaluation metric.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Target Variable: Churn Distribution', fontsize=15, color='#a78bfa')

labels_str = ['No Churn (0)', 'Churn (1)']
bars = axes[0].bar(labels_str, [vc[0], vc[1]], color=PALETTE, width=0.5)
axes[0].set_title('Count'); axes[0].set_ylabel('Number of Customers')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, cnt in zip(bars, [vc[0], vc[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'{cnt:,}', ha='center', fontsize=11)

axes[1].pie([pct[0], pct[1]], labels=labels_str, colors=PALETTE,
            autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='black', linewidth=2))
axes[1].set_title('Percentage')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/step4_churn_distribution.png', dpi=140, bbox_inches='tight')
plt.show()

---
## 📊 Step 5: Numerical Features — Summary Statistics

In [ ]:
stats = df[NUMERICAL].describe().T
stats['median']    = df[NUMERICAL].median()
stats['missing']   = df[NUMERICAL].isnull().sum()
stats['missing_%'] = (df[NUMERICAL].isnull().sum() / len(df) * 100).round(2)
stats[['mean','median','std','min','max','missing_%']].round(2)

### Histograms — Distributions

In [ ]:
n_cols = 3
n_rows = (len(NUMERICAL) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
fig.suptitle('Numerical Features — Distributions', fontsize=15, color='#a78bfa')
for i, feat in enumerate(NUMERICAL):
    ax = axes.flatten()[i]
    data = df[feat].dropna()
    ax.hist(data, bins=50, color='#6C63FF', edgecolor='none', alpha=0.85)
    ax.axvline(data.mean(),   color='#FF6584', lw=1.5, linestyle='--', label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='#F7B731', lw=1.5, linestyle='-',  label=f'Median={data.median():.1f}')
    ax.set_title(feat); ax.legend(fontsize=7, framealpha=0.3)
for j in range(i+1, len(axes.flatten())): axes.flatten()[j].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/step5_numerical_histograms.png', dpi=140, bbox_inches='tight')
plt.show()

### Box Plots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
fig.suptitle('Numerical Features — Box Plots', fontsize=15, color='#a78bfa')
for i, feat in enumerate(NUMERICAL):
    ax = axes.flatten()[i]
    data = df[feat].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75); IQR = Q3 - Q1
    n_out = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
    ax.boxplot(data, patch_artist=True,
               boxprops=dict(facecolor='#6C63FF', color='#a78bfa'),
               medianprops=dict(color='#FF6584', linewidth=2),
               flierprops=dict(marker='.', color='#FF6584', alpha=0.2, markersize=2))
    ax.set_title(f'{feat}\n({n_out:,} outliers)', fontsize=10); ax.set_xticks([])
for j in range(i+1, len(axes.flatten())): axes.flatten()[j].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/step5_numerical_boxplots.png', dpi=140, bbox_inches='tight')
plt.show()

### Outlier Summary (IQR Method)

In [ ]:
outlier_summary = []
for feat in NUMERICAL:
    data = df[feat].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75); IQR = Q3 - Q1
    n_out = ((data < Q1-1.5*IQR) | (data > Q3+1.5*IQR)).sum()
    outlier_summary.append({'Feature': feat, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                             'Outliers': n_out, 'Outlier_%': n_out/len(data)*100})
pd.DataFrame(outlier_summary).set_index('Feature').round(2)

---
## 🏷️ Step 6: Categorical Features Analysis

In [ ]:
for feat in CATEGORICAL:
    vc_ = df[feat].value_counts()
    pct_= df[feat].value_counts(normalize=True)*100
    tbl = pd.DataFrame({'Count': vc_, 'Percentage(%)': pct_.round(2)})
    print(f'\n--- {feat} ---')
    print(tbl.to_string())

In [ ]:
CAT_PALETTE = ['#6C63FF','#FF6584','#43BCCD','#F7B731','#a29bfe','#fd79a8']
n_cols = 2; n_rows = (len(CATEGORICAL) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows*4))
fig.suptitle('Categorical Features — Value Counts', fontsize=15, color='#a78bfa')
for i, feat in enumerate(CATEGORICAL):
    ax = axes.flatten()[i]
    vc_ = df[feat].value_counts()
    bars = ax.bar(vc_.index.astype(str), vc_.values, color=CAT_PALETTE[:len(vc_)], edgecolor='none')
    ax.set_title(feat); ax.tick_params(axis='x', rotation=25)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for bar, cnt in zip(bars, vc_.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vc_.values)*0.01,
                f'{cnt/len(df)*100:.1f}%', ha='center', fontsize=9)
for j in range(i+1, len(axes.flatten())): axes.flatten()[j].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/step6_categorical_countplots.png', dpi=140, bbox_inches='tight')
plt.show()

### Binary Features — Adoption Rates

In [ ]:
binary_df = pd.DataFrame({
    'Feature': BINARY + [TARGET],
    'Rate_1 (%)': [(df[f]==1).mean()*100 for f in BINARY + [TARGET]],
    'Rate_0 (%)': [(df[f]==0).mean()*100 for f in BINARY + [TARGET]],
})
binary_df.set_index('Feature').round(2)

In [ ]:
all_binary = BINARY + [TARGET]
rates = [(df[f]==1).mean()*100 for f in all_binary]
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Binary Features — Adoption Rate (% = 1)', fontsize=14, color='#a78bfa')
bars = ax.barh(all_binary, rates, color='#6C63FF', alpha=0.85)
ax.barh(all_binary, [100-r for r in rates], left=rates, color='#2a2a4a', alpha=0.5)
for bar, rate in zip(bars, rates):
    ax.text(rate+1, bar.get_y()+bar.get_height()/2, f'{rate:.1f}%', va='center', fontsize=10)
ax.set_xlim(0, 115); ax.set_xlabel('Adoption Rate (%)')
ax.axvline(50, color='#FF6584', lw=1, linestyle='--', alpha=0.5, label='50%'); ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/step6_binary_adoption_rates.png', dpi=140, bbox_inches='tight')
plt.show()

---
## ✅ Today's EDA Complete

**Next: Step 7 — Bivariate EDA (Churn vs Every Feature)**

We'll discover which features most strongly predict churn — the core of business insight generation.